# Index-selection Knowledge Indicators with the Context Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kderusso/elasticsearch-labs/blob/kderusso/context-engine-notebooks/notebooks/context-engine/manual-walkthrough/index-metadata-kis.ipynb)

The **Context Engine** gives AI agents structured, pre-computed **Knowledge Indicators (KIs)**, facts they can retrieve in a single call instead of scanning thousands of documents. This interactive notebook will walk you through how to set up a context engine, using the official [Elasticsearch Python client](https://www.elastic.co/guide/en/elasticsearch/client/python-api/current/index.html) and the [Kibana Workflows API](https://www.elastic.co/guide/en/kibana/current/index.html).

For this walkthrough, we index three [BEIR benchmark](https://github.com/beir-cellar/beir) corpora covering distinct domains, profile each into an index-level KI, and show how routing with the Context Engine replaces scatter-gather search.


## Create Elasticsearch environment

This notebook will work on an Elastic Serverless project, or an Elastic Cloud deployment with version 9.5.0 or higher. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?onboarding_token=search&cta=cloud-registration&tech=trial&plcmt=article%20content&pg=search-labs).

Once logged in to your Elastic Cloud account, go to the Create page and select Create project or Create deployment.

## Install packages and import modules

To get started, we'll need to connect to our Elastic deployment using the Python client. Because we're using an Elastic Cloud deployment, we'll use the `cloud_id` to identify our deployment.

In [ ]:
!pip install -q "elasticsearch>=9,<10" datasets requests pyyaml ipywidgets langchain-openai langgraph deepagents

### Initialize the Elasticsearch client

Now we can instantiate the Elasticsearch Python client, providing the Cloud ID and API key for your deployment.

- [Find your Cloud ID](https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id)
- [Create an API key](https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key)

In [ ]:
import json
import time
import base64
import requests
from getpass import getpass
from elasticsearch import Elasticsearch, helpers

ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic API key: ")

client = Elasticsearch(cloud_id=ELASTIC_CLOUD_ID, api_key=ELASTIC_API_KEY)
print(client.info())

Next, we ensure that Kibana is accessible.

In [ ]:
def _urls_from_cloud_id(cloud_id):
    _, encoded = cloud_id.split(":", 1)
    host, es_uuid, kibana_uuid = base64.b64decode(encoded).decode().split("$")
    return f"https://{es_uuid}.{host}", f"https://{kibana_uuid}.{host}"

ES_URL, KIBANA_URL = _urls_from_cloud_id(ELASTIC_CLOUD_ID)

def kbn_request(method, path, *, body=None, internal=False, api_version=None):
    """Call a Kibana REST API and return the parsed JSON response."""
    headers = {
        "Authorization": f"ApiKey {ELASTIC_API_KEY}",
        "kbn-xsrf": "true",
        "Content-Type": "application/json",
    }
    if internal:
        headers["x-elastic-internal-origin"] = "Kibana"
    if api_version:
        headers["elastic-api-version"] = api_version
    resp = requests.request(
        method,
        f"{KIBANA_URL}{path}",
        headers=headers,
        data=json.dumps(body) if body is not None else None,
    )
    resp.raise_for_status()
    return resp.json() if resp.text else {}

status = kbn_request("GET", "/api/status")
print("Kibana status:", status.get("status", {}).get("overall", {}).get("level"))

### Enable the Context Engine feature flag

The Context Engine is gated behind the `agentBuilder:experimentalFeatures` advanced setting. We will enable this flag via the settings API.

> If the cell prints "Could not set feature flag", you may not be able to set this feature flag via the API. Verify that the `agentBuilder:experimentalFeatures` feature flag is toggled ON in Stack Management → Advanced Settings.

In [ ]:
try:
    kbn_request("POST", "/internal/kibana/settings",
                body={"changes": {"agentBuilder:experimentalFeatures": True}}, internal=True)
    print("Enabled feature flag: agentBuilder:experimentalFeatures")
except Exception as e:
    print(f"Could not set feature flag via API: {e}")
    print("Verify that the `agentBuilder:experimentalFeatures` feature flag is toggled ON in Stack Management → Advanced Settings.")

### Connect a language model (OpenRouter)

We use an OpenRouter key with the LangChain `ChatOpenAI` client for all LLM calls — both the before/after answer synthesis and the deep-agent harness at the end. (No EIS / `_inference` API.)

In [ ]:
from langchain_openai import ChatOpenAI

OPENROUTER_API_KEY = getpass("OpenRouter API key: ")
OPENROUTER_MODEL = "anthropic/claude-haiku-4-5"

# Shared model for the before/after answer synthesis (lean token budget for concise answers).
llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    max_tokens=512,
)

def answer_question(question, context_chunks):
    """Synthesize an answer from a list of context strings."""
    ctx = "\n\n---\n\n".join(context_chunks)
    prompt = (
        "Answer the question using only the context provided. "
        "Be concise. If the context does not contain the answer, say so.\n\n"
        f"Context:\n{ctx}\n\nQuestion: {question}"
    )
    return llm.invoke(prompt).content

## Index sample BEIR datasets (BM25 only)

We use three [BEIR benchmark](https://github.com/beir-cellar/beir) corpora — each a distinct domain — indexed BM25-only for speed. We stream 100 documents per dataset from Hugging Face.

| Dataset | Domain | Index name |
|---------|--------|------------|
| [FiQA](https://huggingface.co/datasets/BeIR/fiqa) | Financial Q&A | `beir-fiqa` |
| [NFCorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | Biomedical / nutrition | `beir-nfcorpus` |
| [SciFact](https://huggingface.co/datasets/BeIR/scifact) | Scientific fact-checking | `beir-scifact` |

We attach `_meta.description` and per-field `meta.description` to each index — this human-written context is what the profiling workflow reads.

In [ ]:
from datasets import load_dataset

SAMPLE_DOCS = 100

DATASETS = [
    {
        "hf_dataset": "BeIR/fiqa",
        "hf_config": "corpus",
        "hf_split": "corpus",
        "index_name": "beir-fiqa",
        "meta_description": (
            "FiQA: financial question answering corpus from StackExchange Finance "
            "community posts and web crawls. Covers investments, banking, taxes, "
            "and market analysis. BM25-only index."
        ),
    },
    {
        "hf_dataset": "BeIR/nfcorpus",
        "hf_config": "corpus",
        "hf_split": "corpus",
        "index_name": "beir-nfcorpus",
        "meta_description": (
            "NFCorpus: biomedical information retrieval corpus from NutritionFacts.org. "
            "Contains nutrition science and medical research documents on diet, disease, "
            "and health interventions. BM25-only index."
        ),
    },
    {
        "hf_dataset": "BeIR/scifact",
        "hf_config": "corpus",
        "hf_split": "corpus",
        "index_name": "beir-scifact",
        "meta_description": (
            "SciFact: scientific fact-checking corpus of biomedical research abstracts "
            "used to verify factual claims in peer-reviewed literature. BM25-only index."
        ),
    },
]

for ds in DATASETS:
    idx = ds["index_name"]
    client.indices.delete(index=idx, ignore_unavailable=True)
    client.indices.create(
        index=idx,
        mappings={
            "_meta": {"description": ds["meta_description"]},
            "properties": {
                "title": {"type": "text", "meta": {"description": "Document or article title."}},
                "text": {"type": "text", "meta": {"description": "Full document body text."}},
            },
        },
    )

    corpus = load_dataset(
        ds["hf_dataset"], ds["hf_config"], split=ds["hf_split"], streaming=True
    )

    def actions(corpus, index_name, n):
        for i, row in enumerate(corpus):
            if i >= n:
                break
            yield {
                "_index": index_name,
                "_id": row["_id"],
                "_source": {"title": row.get("title", ""), "text": row.get("text", "")},
            }

    helpers.bulk(client, actions(corpus, idx, SAMPLE_DOCS))
    client.indices.refresh(index=idx)
    print(f"Indexed {client.count(index=idx)['count']} documents into '{idx}'.")

## Ask a question referencing this data

Without index-level KIs there is no way to know which of the three indices is relevant for a given question. The only option is to search all of them and pass the merged results to an LLM.

In [ ]:
QUESTION = "What investment strategies minimize long-term financial risk?"

print("=== Without KIs: scatter-gather across all indices ===")
print()
chunks = []
for ds in DATASETS:
    idx = ds["index_name"]
    resp = client.search(
        index=idx,
        body={
            "query": {"multi_match": {"query": QUESTION, "fields": ["title", "text"]}},
            "size": 3,
            "_source": ["title", "text"],
        },
    )
    for hit in resp["hits"]["hits"]:
        src = hit["_source"]
        chunks.append(f"[from {idx}] {src.get('title', '')}\n{src.get('text', '')[:500]}")

print(f"Gathered {len(chunks)} chunks from {len(DATASETS)} indices.")
print()
for i, chunk in enumerate(chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()
print("=== Answer ===")
print()
print(answer_question(QUESTION, chunks))

## Create a workflow to generate Knowledge Indicators

This workflow profiles each index and writes a routing KI into the Context Engine:

| Step | Type | What it does |
|------|------|--------------|
| `get_mapping` | `elasticsearch.request` | Read the mapping, including `_meta.description` and per-field descriptions. |
| `sample_docs` | `elasticsearch.search` | Pull a few real documents so the profile reflects actual value shapes. |
| `profile_index` | `ai.agent` | Generate a structured index profile as structured output. |
| `sink_index_ki` | `contextEngine.addEntry` | Write the profile into the Context Engine as a KI. |

### GenAI connector (optional)

Leave blank to use Kibana's default GenAI connector, or paste a connector id to pin one.

In [ ]:
LLM_CONNECTOR_ID = input("Kibana GenAI connector id (blank = default connector): ").strip()

In [ ]:
_WORKFLOW_YAML_TEMPLATE = """
version: '1'
name: beir-index-profile-ki
description: Profile an index into an index-selection Knowledge Indicator.
enabled: true
tags:
  - context-engine
  - index-selection
triggers:
  - type: manual
    inputs:
      - name: indexName
        type: string
        required: true
        description: The Elasticsearch index to profile.
steps:
  - name: get_mapping
    type: elasticsearch.request
    with:
      method: GET
      path: '/{{ inputs.indexName }}/_mapping'

  - name: sample_docs
    type: elasticsearch.search
    with:
      index: '{{ inputs.indexName }}'
      size: 3
      query:
        match_all: {}

  - name: profile_index
    type: ai.agent
    connector-id: "__LLM_CONNECTOR_ID__"
    timeout: 120s
    with:
      message: >
        You are a data steward building an INDEX PROFILE for an enterprise data
        catalog. Downstream, an AI agent uses these profiles to decide WHICH
        Elasticsearch index to query for a given user question - this is an
        index-SELECTION aid, not a place to answer the question itself.

        Ground everything in the provided mapping + samples. Never invent fields,
        values, or purpose. If unknown, use an empty string/array. Optimize for
        routing: make it obvious what questions this index can authoritatively
        answer, and what it cannot. Prefer concrete field names and real example
        values over vague phrasing.

        Index name: {{ inputs.indexName }}

        Elasticsearch mapping (JSON):
        {{ steps.get_mapping.output | json }}

        Sample documents (JSON):
        {{ steps.sample_docs.output.hits.hits | map: '_source' | json }}
      schema:
        type: object
        properties:
          display_name:
            type: string
            description: A concise human-readable name for what this index represents (<= 8 words).
          purpose:
            type: string
            description: 2-4 sentences describing what this index stores and its role. PRIMARY semantic surface for matching a question to this index.
          answers_questions:
            type: array
            items:
              type: string
            description: 3-7 representative natural-language questions this index can authoritatively answer.
          does_not_contain:
            type: array
            items:
              type: string
            description: 1-4 things a searcher might wrongly expect here but that live elsewhere, to prevent mis-routing.
          key_fields:
            type: array
            items:
              type: string
            description: 3-10 of the most query-relevant fields as "field_name - what it is".
          when_to_use:
            type: string
            description: A single crisp routing heuristic - when should an agent pick THIS index? (<= 30 words).
          example_esql:
            type: string
            description: One realistic, runnable ES|QL query against this index answering one of answers_questions.
        required:
          - display_name
          - purpose
          - answers_questions
          - key_fields
          - when_to_use

  - name: sink_index_ki
    type: contextEngine.addEntry
    with:
      originId: '{{ inputs.indexName }}'
      attachmentType: corpus_entry
      action: upsert
      chunks:
        - type: corpus_entry
          title: '{{ steps.profile_index.output.structured_output.display_name | default: inputs.indexName }}'
          content: >
            === SOURCE / PROVENANCE ===
            This is an INDEX PROFILE for routing/index-selection.
            Backing Elasticsearch index: {{ inputs.indexName }}
            Inspect it directly with ES|QL:
            FROM {{ inputs.indexName }} | LIMIT 10
            === WHAT THIS INDEX IS ===
            {{ steps.profile_index.output.structured_output.purpose }}
            Questions this index can answer: {{ steps.profile_index.output.structured_output.answers_questions | join: " | " }}
            When to use this index: {{ steps.profile_index.output.structured_output.when_to_use }}
            Example query:
            {{ steps.profile_index.output.structured_output.example_esql }}
          description: >
            Index profile: {{ steps.profile_index.output.structured_output.display_name }}.
            Does NOT contain: {{ steps.profile_index.output.structured_output.does_not_contain | join: "; " }}.
            Key fields: {{ steps.profile_index.output.structured_output.key_fields | join: "; " }}.

  - name: output_result
    type: workflow.output
    with:
      indexName: '{{ inputs.indexName }}'
      display_name: '{{ steps.profile_index.output.structured_output.display_name }}'
      when_to_use: '{{ steps.profile_index.output.structured_output.when_to_use }}'
"""

# Pin a connector if one was supplied.
if LLM_CONNECTOR_ID:
    WORKFLOW_YAML = _WORKFLOW_YAML_TEMPLATE.replace("__LLM_CONNECTOR_ID__", LLM_CONNECTOR_ID)
else:
    WORKFLOW_YAML = "\n".join(
        line for line in _WORKFLOW_YAML_TEMPLATE.splitlines() if "__LLM_CONNECTOR_ID__" not in line
    )

print(WORKFLOW_YAML)

In [ ]:
WF_API_VERSION = "2023-10-31"
TERMINAL_STATES = {"completed", "failed", "cancelled", "timed_out", "skipped"}

def create_workflow(yaml_str):
    return kbn_request("POST", "/api/workflows/workflow",
                       body={"yaml": yaml_str}, api_version=WF_API_VERSION)

def run_workflow(workflow_id, inputs):
    return kbn_request("POST", f"/api/workflows/workflow/{workflow_id}/run",
                       body={"inputs": inputs}, api_version=WF_API_VERSION)

def wait_for_execution(execution_id, timeout=300, interval=3):
    deadline = time.time() + timeout
    while time.time() < deadline:
        ex = kbn_request("GET", f"/api/workflows/executions/{execution_id}?includeOutput=true",
                         api_version=WF_API_VERSION)
        if ex["status"] in TERMINAL_STATES:
            return ex
        time.sleep(interval)
    raise TimeoutError(f"Execution {execution_id} did not finish within {timeout}s")

wf = create_workflow(WORKFLOW_YAML)
workflow_id = wf["id"]
print("Created workflow:", workflow_id)

This workflow will generate and index Knowledge Indicators. This step may take a few minutes to run.

In [ ]:
execution_results = {}

for ds in DATASETS:
    idx = ds["index_name"]
    run_resp = run_workflow(workflow_id, {"indexName": idx})
    exec_id = run_resp["workflowExecutionId"]
    print(f"Profiling '{idx}' — execution: {exec_id}")

    result = wait_for_execution(exec_id)
    execution_results[idx] = result
    print(f"  Status: {result['status']}")

    if result["status"] == "completed":
        output = result.get("context", {}).get("output", {})
        print(f"  Profile: {output.get('display_name', '—')} — {output.get('when_to_use', '—')}")
    else:
        if result.get("error"):
            print("  Error:", result["error"].get("message", result["error"]))
        for step in result.get("stepExecutions", []):
            if step.get("status") == "failed":
                print("  Failed step:", step.get("stepId"), "->", json.dumps(step.get("error")))

## Ask the same question using Knowledge Indicators

With index-level KIs in the Context Engine, one `get_context` call identifies the right index. We show the routing KIs returned, then search only the top-ranked index and synthesize an answer from those results.

In [ ]:
def get_context(query, size=5, types=None):
    body = {
        "query": query,
        "size": size,
        "fields": ["content", "description", "tags", "references"],
    }
    if types:
        body["filters"] = {"types": types}
    return kbn_request("POST", "/internal/agent_context_layer/sml/_search",
                       body=body, internal=True)

response = get_context(QUESTION, types=["corpus_entry"])
ki_results = response.get("results", [])

print(f"Knowledge Indicators retrieved: {len(ki_results)}\n")
for ki in ki_results:
    print(f"  [{ki.get('origin', {}).get('uri', '?')}] {ki.get('title')}")
    print(f"  {ki.get('description', '')[:150]}")
    print()

chunks = [f"{r.get('title', '')}\n{r.get('content', '')}" for r in ki_results]
print("=== Answer from Context Engine KIs ===")
print()
print(answer_question(QUESTION, chunks))

### Fetch a single KI by id via the SML API

The `_search` endpoint ranks KIs; the SML API also lets you fetch one entry **by id**. We take the top KI from the search above and issue `GET /internal/agent_context_layer/sml/{id}` to display its full stored record.

In [ ]:
# Pick one KI (the top search hit) and fetch its full record directly by id via the SML API.
hits = get_context(QUESTION, size=1, types=["corpus_entry"])["results"]
if not hits:
    print("No KIs found — run the workflow cells above first.")
else:
    ki_id = hits[0]["id"]
    print(f"GET /internal/agent_context_layer/sml/{ki_id}\n")
    ki = kbn_request("GET", f"/internal/agent_context_layer/sml/{ki_id}", internal=True)
    print(json.dumps(ki, indent=2))

## Consume the KIs from a deep agent

The Context Engine is meant to be **consumed by an agent**. Here we wrap `get_context` as a tool over the index-profile KIs and let a [`deepagents`](https://pypi.org/project/deepagents/) agent make the routing decision itself: it searches the profiles and recommends which index to query.

In [ ]:
from langchain_core.tools import tool
from deepagents import create_deep_agent

# Roomier token budget for the planning/tool-calling agent (llm stays lean for synthesis).
agent_llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    max_tokens=4096,
)

@tool
def search_context(query: str) -> str:
    """Search the Context Engine for Knowledge Indicators relevant to the question.

    Pass a focused natural-language query. Returns the most relevant KIs with their
    title and content (each includes provenance for the backing index/document).
    """
    results = get_context(query, size=5, types=["corpus_entry"])["results"]
    if not results:
        return "No relevant context found in the Context Engine."
    return "\n\n".join(f"[{r['title']}]\n{r.get('content', '')}" for r in results)

SYSTEM_PROMPT = """You are a research assistant. When a question depends on specific facts or on
which dataset holds the answer, call `search_context` to retrieve relevant Knowledge Indicators,
then answer grounded in what it returns, citing the KI titles you used."""

agent = create_deep_agent(
    model=agent_llm,
    tools=[search_context],
    system_prompt=SYSTEM_PROMPT,
)

result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})
print(result["messages"][-1].content)

## Clean up

In [ ]:
index_names = {ds["index_name"] for ds in DATASETS}
for hit in get_context("index profile", size=50, types=["corpus_entry"])["results"]:
    uri = hit.get("origin", {}).get("uri", "")
    if uri.split("://", 1)[-1] in index_names:
        kbn_request("DELETE", f"/internal/agent_context_layer/sml/{hit['id']}", internal=True)
        print("Deleted KI:", hit["id"], "—", hit.get("title"))

try:
    kbn_request("DELETE", f"/api/workflows/workflow/{workflow_id}", api_version=WF_API_VERSION)
    print("Deleted workflow:", workflow_id)
except Exception as e:
    print("Workflow delete skipped:", e)

for ds in DATASETS:
    client.indices.delete(index=ds["index_name"], ignore_unavailable=True)
    print("Deleted index:", ds["index_name"])